# R05 — per-segment replay evidence

Real-cloud replay with frozen references and independently calibrated arms. Nulls, injection failures and adverse results remain visible. This notebook reads the sealed R05 run; it does not retune or open H.

In [ ]:
import os, sys, json, csv
from pathlib import Path
from IPython.display import display, Markdown, Image
root = Path.cwd()
sys.path.insert(0, str(root / 'src'))
from valid_point.evaluation.real import require_prediction_seal
from valid_point.evaluation.statistics import read_records
run = Path(os.environ['VP_RUN_DIR'])
seal = json.loads((run/'prediction_seal.json').read_text())
require_prediction_seal(run/'predictions', seal)
print('Run:', run.name, '\nFreeze:', json.loads((run/'protocol.json').read_text())['freeze_sha256'])
print('Units: counts points/cell/frame; geometry metres; clocks integer ns; plots seconds; scores dimensionless.')
print('All four folds sealed before evaluator joins. Labels are unused. H is untouched.')


In [ ]:
segment=os.environ['VP_SEGMENT']
fold='F'+segment[-1]
intake=json.loads((run/'intake'/(segment+'.json')).read_text())
display(Markdown('## Input/schema/frame/time checks'))
display(intake)


In [ ]:
display(Markdown('## Actual clean raw records and calibration support'))
raw_path=run/'clean_raw'/(segment+'.jsonl.gz')
if raw_path.exists():
    raw=read_records(raw_path)
    display(raw[:3])
    unknown=[r for r in raw if r['geometry']['distance_m'] is None or r['density']['counts'] is None]
    print('Raw events:',len(raw),'unknown factor events:',len(unknown))
    display(unknown[:3])
threshold_path=run/'calibration'/fold/'thresholds.json'
if threshold_path.exists():
    thresholds=json.loads(threshold_path.read_text())
    display([{k:v for k,v in t.items() if k not in ('calibration_record_ids','calibration_raw_ids')} for t in thresholds])
else:
    display(Markdown('Fold unusable: explicit failed study cells are retained.'))


In [ ]:
display(Markdown('## Clean, benign, severity, coverage, delay and recovery cells'))
for name in ('T_R05_metrics','T_R05_coverage','T_R05_failures','T_R05_paired_effects'):
    with (run/(name+'.csv')).open() as f: rows=[r for r in csv.DictReader(f) if r['segment']==segment]
    print(name, 'full rows:',len(rows), 'CSV/Markdown:',run/(name+'.csv'))
    display(rows[:12])


In [ ]:
display(Markdown('## Geometry and record timelines: all five streams, units and unknown gaps'))
figures=sorted(run.glob('F_R05_*_'+segment+'_*.png'))
if not figures:
    display(Markdown('No valid geometry/timeline exists for this failed segment. No plot is fabricated.'))
for figure in figures: display(Image(filename=str(figure)))


In [ ]:
display(Markdown('## Censoring and comparable support'))
for name in ('T_R05_episode_outcomes','T_R05_exposure_time','T_R05_common_support'):
    with (run/(name+'.csv')).open() as f: rows=[r for r in csv.DictReader(f) if r['segment']==segment]
    print(name, len(rows), 'full rows; delay in seconds, right censoring retained; no interval spans an unknown')
    display(rows[:8])


In [ ]:
display(Markdown('## Comparison disposition'))
display(Markdown((run/'gate.md').read_text()))
require_prediction_seal(run/'predictions',seal)
